# Akıllı Otopark — Model Eğitimi (Google Colab)

Bu not defteri `training/` klasöründeki scriptleri Colab GPU'sunda çalıştırır:
1. Plaka tespit modeli (YOLOv8n, tek sınıf) → `plate_detector_tr.pt`
2. Karakter okuyucu (YOLOv8n, 36 sınıf, imgsz=320) → `char_reader.pt` + `char_classes.txt`
3. Değerlendirme ve ONNX dışa aktarım

**Önce:** Çalışma zamanı → Çalışma zamanı türünü değiştir → **T4 GPU**.

In [ ]:
!nvidia-smi -L
!pip -q install "ultralytics>=8.3" roboflow onnx onnxruntime pyyaml

## 1. `training/` klasörünü Colab'a getirin
Seçenek A: Projeyi Google Drive'a yükleyip bağlayın. Seçenek B: `training` klasörünü zip'leyip sol paneldeki Dosyalar'a sürükleyin.

In [ ]:
# Seçenek A — Google Drive
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/OtoparkSistemi/training

# Seçenek B — zip yüklediyseniz (yukarıdaki iki satırı yorum yapın):
# !unzip -q /content/training.zip -d /content && %cd /content/training

## 2. Veri setlerini indirin
Roboflow API anahtarınızı (https://app.roboflow.com → Settings → Roboflow API) girin. Anahtar not defterine kaydedilmez.

In [ ]:
import os, getpass
if not os.environ.get('ROBOFLOW_API_KEY'):
    os.environ['ROBOFLOW_API_KEY'] = getpass.getpass('ROBOFLOW_API_KEY: ')
!python indir_veri.py --set hepsi

Test kümesi yoksa %70/20/10 bölün (ör. plaka tespiti için):

In [ ]:
# !python bolme.py --kaynak data/plaka_tespit --hedef data/plaka_tespit_bolunmus
# Sonraki hücrelerde --data yolunu data/plaka_tespit_bolunmus/data.yaml yapın.

## 3. Plaka tespit modeli (≈ 100 epoch, T4'te ~1 saat)

In [ ]:
!python egit_plaka_tespit.py --data data/plaka_tespit/data.yaml --epochs 100 --patience 30 --imgsz 640 --batch 16 --device 0 --workers 2

## 4. Karakter okuyucu (imgsz=320)

In [ ]:
!python egit_karakter_okuyucu.py --data data/karakter/data.yaml --epochs 100 --patience 30 --imgsz 320 --batch 32 --device 0 --workers 2 --kopyala

## 5. Değerlendirme

In [ ]:
# Karakter okuyucunun test kümesinde plaka bazında doğruluğu (tüm karakterler doğru oranı)
!python degerlendir.py --karakter-test data/karakter --model runs/detect/char-reader/weights/best.pt --siniflar char_classes.txt
# Plaka tespit mAP
!python degerlendir.py --tespit-test data/plaka_tespit/data.yaml --model runs/detect/plate-detector-tr/weights/best.pt

## 6. ONNX dışa aktarım (isteğe bağlı, CPU'da hızlı çıkarım)

In [ ]:
!python onnx_aktar.py --model runs/detect/char-reader/weights/best.pt --imgsz 320
!python onnx_aktar.py --model runs/detect/plate-detector-tr/weights/best.pt --imgsz 640

## 7. Modelleri indirin
Dosyaları bilgisayarınızda `vision-service/models/` altına koyun:
- `char-reader/weights/best.pt` → `char_reader.pt` (+ `char_classes.txt`)
- `plate-detector-tr/weights/best.pt` → `plate_detector_tr.pt` (hazır modeli değiştirmek istiyorsanız)

In [ ]:
import shutil
shutil.copy('runs/detect/char-reader/weights/best.pt', 'char_reader.pt')
from google.colab import files
files.download('char_reader.pt')
files.download('char_classes.txt')